# model validation

Compare human labels with model predictions using matched record keys. Requires the manually labeled v3 workbook; never substitute generated blank labels.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# AEGEAN AIRLINES — MODEL VALIDATION
# Human Ground Truth vs Hugging Face, Gemini and Haiku
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    jaccard_score
)

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = PROJECT_ROOT

NOTEBOOK_DIR = PROJECT_DIR / "Python Code"

DATA_DIR = HF_DIR

LLM_DIR = LLM_OUTPUT_DIR

VALIDATION_FILE = (
    LLM_DIR / "model_validation_100_reviewsv3.xlsx"
)

HF_FILE = DATA_DIR / "df_final_absa.csv"

GEMINI_FILE = LLM_DIR / "df_final_absa_gemini.csv"

HAIKU_FILE = LLM_DIR / "df_final_absa_haiku.csv"
if not HAIKU_FILE.exists():
    HAIKU_FILE = LLM_DIR / "Pilot" / "df_final_absa_haiku.csv"

OUTPUT_DIR = LLM_DIR / "validation_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# VALIDATION SETTINGS
# ============================================================

START_DATE = "2024-01-01"

APPLY_DATE_FILTER = True
EXCLUDE_LONG_HAUL = True
EXCLUDE_FIRST_CLASS = True

# Only compare models on identical observations
USE_MATCHED_SAMPLE = True

# Automatically include Haiku if its CSV exists
INCLUDE_HAIKU = True

# ============================================================
# SENTIMENT LABELS
# ============================================================

OVERALL_LABELS = [
    "positive",
    "negative",
    "neutral",
    "mixed"
]

ASPECT_LABELS = [
    "positive",
    "negative",
    "neutral",
    "mixed",
    "not_mentioned"
]

BINARY_LABELS = [0, 1]

# ============================================================
# 25 CLASSIFICATION FEATURES
# ============================================================

FEATURES = [
    "long_haul",
    "first_class",
    "cancellation",
    "delay",
    "boarding",
    "baggage",
    "economy_class",
    "business_class",
    "international",
    "domestic",
    "customer_service",
    "cabin_crew",
    "ground_staff",
    "complaint_resolution",
    "seat_comfort",
    "food_beverage",
    "cabin_cleanliness",
    "entertainment_wifi",
    "booking_experience",
    "check_in",
    "airport_experience",
    "value_for_money",
    "pricing_fees",
    "flight_disruption",
    "overall_experience"
]

JOURNEY_CONTEXT_FEATURES = [
    "long_haul",
    "first_class",
    "economy_class",
    "business_class",
    "international",
    "domestic"
]

EXPERIENCE_FEATURES = [
    feature for feature in FEATURES
    if feature not in JOURNEY_CONTEXT_FEATURES
]

MODEL_COLORS = {
    "Hugging Face": "#607D8B",
    "Gemini": "#4285F4",
    "Haiku": "#D97757"
}

print("Configuration loaded successfully.")
print("Validation file:", VALIDATION_FILE)
print("Output directory:", OUTPUT_DIR)
print("Number of features:", len(FEATURES))

In [ ]:
# ============================================================
# LOAD VALIDATION WORKBOOK
# ============================================================

if not VALIDATION_FILE.exists():
    raise FileNotFoundError(VALIDATION_FILE)

workbook = pd.ExcelFile(VALIDATION_FILE)

print("Available Excel sheets:")
print(workbook.sheet_names)

df_manual_overall = pd.read_excel(
    workbook,
    sheet_name="Overall Validation"
)

df_manual_aspect = pd.read_excel(
    workbook,
    sheet_name="Aspect Validation"
)

# Normalize column names
for df in [df_manual_overall, df_manual_aspect]:

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

print("\nOverall validation shape:")
print(df_manual_overall.shape)

print("\nAspect validation shape:")
print(df_manual_aspect.shape)

print("\nOverall validation columns:")
print(df_manual_overall.columns.tolist())

print("\nAspect validation columns:")
print(df_manual_aspect.columns.tolist())


# ============================================================
# LOAD MODEL PREDICTIONS
# ============================================================

def load_model_file(path):

    """Return or process load model file."""
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        low_memory=False,
        dtype={"record_key": str}
    )

    df.columns = df.columns.str.strip()

    if "record_key" not in df.columns:
        raise KeyError(f"record_key missing from {path.name}")

    if df["record_key"].isna().any():
        raise ValueError(f"Missing record keys in {path.name}")

    if df["record_key"].duplicated().any():
        raise ValueError(f"Duplicate record keys in {path.name}")

    return df


df_hf = load_model_file(HF_FILE)
df_gemini = load_model_file(GEMINI_FILE)

model_datasets = {
    "Hugging Face": df_hf,
    "Gemini": df_gemini
}

if INCLUDE_HAIKU and HAIKU_FILE.exists():

    df_haiku = load_model_file(HAIKU_FILE)

    model_datasets["Haiku"] = df_haiku

else:
    print("\nHaiku not included: disabled or file unavailable.")

print("\n" + "=" * 65)
print("MODEL DATASETS")
print("=" * 65)

for model, df in model_datasets.items():

    print(
        f"{model:20s} | "
        f"Reviews: {len(df):,} | "
        f"Columns: {df.shape[1]}"
    )

In [ ]:
# ============================================================
# NORMALIZATION FUNCTIONS
# ============================================================

def normalize_sentiment(series):

    """Return or process normalize sentiment."""
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[\s-]+", "_", regex=True)
        .replace("", pd.NA)
    )


def normalize_binary(series):

    """Return or process normalize binary."""
    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "yes": "1",
            "no": "0",
            "true": "1",
            "false": "0",
            "mentioned": "1",
            "not_mentioned": "0",
            "not mentioned": "0"
        })
    )

    return pd.to_numeric(
        values,
        errors="coerce"
    )


def require_columns(df, columns, name):

    """Return or process require columns."""
    missing = [
        col for col in columns
        if col not in df.columns
    ]

    if missing:
        raise KeyError(
            f"{name}: missing columns {missing}\n"
            f"Available columns: {df.columns.tolist()}"
        )


# Support common alternative feature-column names
if "feature" not in df_manual_aspect.columns:

    for candidate in ["aspect", "aspect_name", "feature_name"]:

        if candidate in df_manual_aspect.columns:

            df_manual_aspect = df_manual_aspect.rename(
                columns={candidate: "feature"}
            )

            break


require_columns(
    df_manual_overall,
    ["record_key", "manual_overall_sentiment"],
    "Overall Validation"
)

require_columns(
    df_manual_aspect,
    [
        "record_key",
        "feature",
        "manual_classification",
        "manual_sentiment"
    ],
    "Aspect Validation"
)

# Normalize record keys
for df in [df_manual_overall, df_manual_aspect]:

    df["record_key"] = (
        df["record_key"]
        .astype("string")
        .str.strip()
    )

    if df["record_key"].isna().any():
        raise ValueError("Missing validation record keys.")

df_manual_aspect["feature"] = (
    df_manual_aspect["feature"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[\s-]+", "_", regex=True)
)

# Check duplicates
if df_manual_overall["record_key"].duplicated().any():

    raise ValueError(
        "Duplicate reviews in Overall Validation."
    )

if df_manual_aspect.duplicated(
    ["record_key", "feature"]
).any():

    raise ValueError(
        "Duplicate review-feature pairs in Aspect Validation."
    )

# Normalize manual labels
df_manual_overall["manual_overall_sentiment"] = (
    normalize_sentiment(
        df_manual_overall["manual_overall_sentiment"]
    )
)

df_manual_aspect["manual_classification"] = (
    normalize_binary(
        df_manual_aspect["manual_classification"]
    )
)

df_manual_aspect["manual_sentiment"] = (
    normalize_sentiment(
        df_manual_aspect["manual_sentiment"]
    )
)

# ============================================================
# CHECK INVALID ANNOTATIONS
# ============================================================

invalid_overall = df_manual_overall.loc[
    df_manual_overall["manual_overall_sentiment"].notna()
    & ~df_manual_overall["manual_overall_sentiment"].isin(
        OVERALL_LABELS
    )
]

invalid_binary = df_manual_aspect.loc[
    df_manual_aspect["manual_classification"].notna()
    & ~df_manual_aspect["manual_classification"].isin(
        BINARY_LABELS
    )
]

invalid_sentiment = df_manual_aspect.loc[
    df_manual_aspect["manual_sentiment"].notna()
    & ~df_manual_aspect["manual_sentiment"].isin(
        ASPECT_LABELS
    )
]

invalid_features = df_manual_aspect.loc[
    ~df_manual_aspect["feature"].isin(FEATURES)
]

# Human annotations must be internally consistent
inconsistent_manual = df_manual_aspect.loc[
    (
        (df_manual_aspect["manual_classification"] == 0)
        & df_manual_aspect["manual_sentiment"].isin(
            OVERALL_LABELS
        )
    )
    |
    (
        (df_manual_aspect["manual_classification"] == 1)
        & (
            df_manual_aspect["manual_sentiment"]
            == "not_mentioned"
        )
    )
]

annotation_checks = {
    "Invalid overall labels": invalid_overall,
    "Invalid binary labels": invalid_binary,
    "Invalid aspect sentiments": invalid_sentiment,
    "Unknown features": invalid_features,
    "Inconsistent annotations": inconsistent_manual
}

for name, issues in annotation_checks.items():

    print(f"{name}: {len(issues)}")

    if not issues.empty:
        display(issues.head(20))

if any(
    not issues.empty
    for issues in annotation_checks.values()
):

    raise ValueError(
        "Correct the annotation issues in the Excel file "
        "before calculating validation metrics."
    )

# ============================================================
# ANNOTATION COVERAGE
# ============================================================

print("\n" + "=" * 65)
print("HUMAN ANNOTATION COVERAGE")
print("=" * 65)

print(
    "Unique reviews:",
    df_manual_overall["record_key"].nunique()
)

print(
    "Overall sentiment annotations:",
    df_manual_overall["manual_overall_sentiment"]
    .notna().sum()
)

print(
    "Aspect identification annotations:",
    df_manual_aspect["manual_classification"]
    .notna().sum()
)

print(
    "Aspect sentiment annotations:",
    df_manual_aspect["manual_sentiment"]
    .notna().sum()
)

print("\nManual overall sentiment distribution:")

display(
    df_manual_overall["manual_overall_sentiment"]
    .value_counts(dropna=False)
)

print("\nManual aspect sentiment distribution:")

display(
    df_manual_aspect["manual_sentiment"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# BUILD VALIDATION SAMPLE
# ============================================================

metadata_columns = [
    "record_key",
    "airline",
    "source",
    "review_date",
    "long_haul",
    "first_class"
]

require_columns(
    df_hf,
    metadata_columns,
    "Hugging Face dataset"
)

metadata = df_hf[metadata_columns].copy()

metadata["review_date"] = pd.to_datetime(
    metadata["review_date"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

metadata["long_haul"] = normalize_binary(
    metadata["long_haul"]
)

metadata["first_class"] = normalize_binary(
    metadata["first_class"]
)

validation_keys = set(
    df_manual_overall["record_key"]
)

metadata = metadata.loc[
    metadata["record_key"].isin(validation_keys)
].copy()

filter_audit = []


def record_filter(stage, df):

    """Return or process record filter."""
    filter_audit.append({
        "stage": stage,
        "reviews": df["record_key"].nunique()
    })


record_filter("Original matched validation sample", metadata)

# Exclude dates before 2024
if APPLY_DATE_FILTER:

    metadata = metadata.loc[
        metadata["review_date"].ge(
            pd.Timestamp(START_DATE)
        )
    ].copy()

    record_filter("From 2024 onwards", metadata)

# Exclude explicit long-haul only
if EXCLUDE_LONG_HAUL:

    metadata = metadata.loc[
        metadata["long_haul"].ne(1)
        | metadata["long_haul"].isna()
    ].copy()

    record_filter("Exclude long-haul", metadata)

# Exclude explicit first class only
if EXCLUDE_FIRST_CLASS:

    metadata = metadata.loc[
        metadata["first_class"].ne(1)
        | metadata["first_class"].isna()
    ].copy()

    record_filter("Exclude first class", metadata)

# Match model datasets
eligible_keys = set(metadata["record_key"])

if USE_MATCHED_SAMPLE:

    for model, df in model_datasets.items():

        eligible_keys &= set(df["record_key"])

        filter_audit.append({
            "stage": f"Matched with {model}",
            "reviews": len(eligible_keys)
        })

metadata = metadata.loc[
    metadata["record_key"].isin(eligible_keys)
].copy()

df_validation_overall = (
    df_manual_overall[
        ["record_key", "manual_overall_sentiment"]
    ]
    .merge(
        metadata,
        on="record_key",
        how="inner",
        validate="one_to_one"
    )
)

df_validation_aspect = (
    df_manual_aspect[
        [
            "record_key",
            "feature",
            "manual_classification",
            "manual_sentiment"
        ]
    ]
    .merge(
        metadata,
        on="record_key",
        how="inner",
        validate="many_to_one"
    )
)

df_filter_audit = pd.DataFrame(filter_audit)

display(df_filter_audit)

print("\nFinal eligible reviews:", len(df_validation_overall))

print(
    "Final eligible review-feature pairs:",
    len(df_validation_aspect)
)

print("\nValidation sample by airline:")

display(
    df_validation_overall["airline"]
    .value_counts()
    .to_frame("reviews")
)

print("\nValidation sample by year:")

display(
    df_validation_overall["review_date"]
    .dt.year.value_counts()
    .sort_index()
    .to_frame("reviews")
)

if df_validation_overall.empty:

    raise ValueError(
        "No validation reviews remain after filtering."
    )

In [ ]:
# ============================================================
# CELL 5 — MERGE MODEL PREDICTIONS WITH HUMAN ANNOTATIONS
# ============================================================

# ============================================================
# MODEL COLUMN MAPPING
# ============================================================

MODEL_CONFIG = {
    "Hugging Face": {
        "overall": "overall_sentiment",
        "binary": "{feature}",
        "sentiment": "{feature}_sentiment"
    },
    "Gemini": {
        "overall": "gemini_overall_sentiment",
        "binary": "{feature}_llm",
        "sentiment": "{feature}_sentiment_llm"
    },
    "Haiku": {
        "overall": "overall_sentiment_llm_metadata",
        "binary": "{feature}_llm",
        "sentiment": "{feature}_sentiment_llm"
    }
}

model_slugs = {
    "Hugging Face": "hf",
    "Gemini": "gemini",
    "Haiku": "haiku"
}

# ============================================================
# RESET PREVIOUSLY MERGED PREDICTIONS
# Makes this cell safe to rerun
# ============================================================

df_validation_overall = (
    df_manual_overall[
        ["record_key", "manual_overall_sentiment"]
    ]
    .merge(
        metadata,
        on="record_key",
        how="inner",
        validate="one_to_one"
    )
)

df_validation_aspect = (
    df_manual_aspect[
        [
            "record_key",
            "feature",
            "manual_classification",
            "manual_sentiment"
        ]
    ]
    .merge(
        metadata,
        on="record_key",
        how="inner",
        validate="many_to_one"
    )
)

print("Validation dataframes reset successfully.")

# ============================================================
# VERIFY ALL MODEL SCHEMAS BEFORE MERGING
# ============================================================

for model, df_model in model_datasets.items():

    config = MODEL_CONFIG[model]

    required_columns = [config["overall"]]

    for feature in FEATURES:

        required_columns.extend([
            config["binary"].format(feature=feature),
            config["sentiment"].format(feature=feature)
        ])

    require_columns(
        df_model,
        required_columns,
        model
    )

    print(
        f"{model:15s} | "
        f"All {len(required_columns)} prediction columns verified"
    )

# ============================================================
# MERGE MODEL PREDICTIONS
# ============================================================

for model, df_model in model_datasets.items():

    config = MODEL_CONFIG[model]
    slug = model_slugs[model]

    print(f"\nProcessing {model}...")

    # --------------------------------------------------------
    # OVERALL SENTIMENT PREDICTIONS
    # --------------------------------------------------------

    overall_predictions = df_model[
        ["record_key", config["overall"]]
    ].copy()

    overall_predictions = overall_predictions.rename(
        columns={
            config["overall"]: f"{slug}_overall"
        }
    )

    overall_predictions[f"{slug}_overall"] = (
        normalize_sentiment(
            overall_predictions[f"{slug}_overall"]
        )
    )

    df_validation_overall = (
        df_validation_overall.merge(
            overall_predictions,
            on="record_key",
            how="left",
            validate="one_to_one"
        )
    )

    # --------------------------------------------------------
    # ASPECT PREDICTIONS IN LONG FORMAT
    # --------------------------------------------------------

    df_sample = df_model.loc[
        df_model["record_key"].isin(eligible_keys)
    ].copy()

    aspect_frames = []

    for feature in FEATURES:

        binary_col = config["binary"].format(
            feature=feature
        )

        sentiment_col = config["sentiment"].format(
            feature=feature
        )

        temp = df_sample[
            ["record_key", binary_col, sentiment_col]
        ].copy()

        temp.columns = [
            "record_key",
            f"{slug}_binary",
            f"{slug}_sentiment"
        ]

        temp["feature"] = feature

        temp[f"{slug}_binary"] = normalize_binary(
            temp[f"{slug}_binary"]
        )

        temp[f"{slug}_sentiment"] = normalize_sentiment(
            temp[f"{slug}_sentiment"]
        )

        aspect_frames.append(temp)

    aspect_predictions = pd.concat(
        aspect_frames,
        ignore_index=True
    )

    df_validation_aspect = (
        df_validation_aspect.merge(
            aspect_predictions,
            on=["record_key", "feature"],
            how="left",
            validate="one_to_one"
        )
    )

    print(
        "Overall predictions:",
        df_validation_overall[
            f"{slug}_overall"
        ].notna().sum()
    )

    print(
        "Binary predictions:",
        df_validation_aspect[
            f"{slug}_binary"
        ].notna().sum()
    )

    print(
        "Aspect sentiment predictions:",
        df_validation_aspect[
            f"{slug}_sentiment"
        ].notna().sum()
    )

# ============================================================
# PREDICTION QUALITY AND COVERAGE AUDIT
# ============================================================

prediction_audit = []

for model in model_datasets:

    slug = model_slugs[model]

    checks = [
        (
            "Overall sentiment",
            df_validation_overall[f"{slug}_overall"],
            OVERALL_LABELS
        ),
        (
            "Aspect identification",
            df_validation_aspect[f"{slug}_binary"],
            BINARY_LABELS
        ),
        (
            "Aspect sentiment",
            df_validation_aspect[f"{slug}_sentiment"],
            ASPECT_LABELS
        )
    ]

    for task, values, allowed_labels in checks:

        valid = values.isin(allowed_labels)

        invalid = (
            values.notna()
            & ~valid
        )

        prediction_audit.append({
            "model": model,
            "task": task,
            "expected_predictions": len(values),
            "valid_predictions": int(valid.sum()),
            "missing_predictions": int(
                values.isna().sum()
            ),
            "invalid_predictions": int(invalid.sum()),
            "coverage_pct": (
                valid.mean() * 100
                if len(values) else np.nan
            )
        })

        if invalid.any():

            print(
                f"\nWARNING: {model} — {task}"
            )

            print("Unexpected labels:")

            display(
                values.loc[invalid].value_counts()
            )

df_prediction_audit = pd.DataFrame(
    prediction_audit
)

# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("MODEL PREDICTIONS MERGED SUCCESSFULLY")
print("=" * 70)

print(
    "Overall validation shape:",
    df_validation_overall.shape
)

print(
    "Aspect validation shape:",
    df_validation_aspect.shape
)

print("\nPREDICTION COVERAGE AUDIT")

display(df_prediction_audit.round(3))

# ============================================================
# OVERALL SENTIMENT DISTRIBUTIONS
# ============================================================

overall_distribution = {
    "Human": df_validation_overall[
        "manual_overall_sentiment"
    ].value_counts()
}

for model in model_datasets:

    slug = model_slugs[model]

    overall_distribution[model] = (
        df_validation_overall[
            f"{slug}_overall"
        ].value_counts()
    )

df_overall_distribution = (
    pd.DataFrame(overall_distribution)
    .fillna(0)
    .astype(int)
)

print("\nOVERALL SENTIMENT DISTRIBUTIONS")

display(df_overall_distribution)

# ============================================================
# PREVIEW FINAL DATAFRAMES
# ============================================================

print("\nOverall sentiment validation:")

display(df_validation_overall.head())

print("\nAspect validation:")

display(df_validation_aspect.head())

In [ ]:
# ============================================================
# CLASSIFICATION METRICS
# ============================================================

def calculate_metrics(y_true, y_pred, labels):

    """Return or process calculate metrics."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)

    if n == 0:
        return {"n": 0}

    observed_labels = [
        label for label in labels
        if label in y_true or label in y_pred
    ]

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        kappa = cohen_kappa_score(
            y_true,
            y_pred,
            labels=labels
        )

        metrics = {
            "n": n,

            "accuracy": accuracy_score(
                y_true, y_pred
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),

            "precision_macro": precision_score(
                y_true,
                y_pred,
                labels=observed_labels,
                average="macro",
                zero_division=0
            ),

            "recall_macro": recall_score(
                y_true,
                y_pred,
                labels=observed_labels,
                average="macro",
                zero_division=0
            ),

            "f1_macro": f1_score(
                y_true,
                y_pred,
                labels=observed_labels,
                average="macro",
                zero_division=0
            ),

            "f1_weighted": f1_score(
                y_true,
                y_pred,
                labels=observed_labels,
                average="weighted",
                zero_division=0
            ),

            "f1_micro": f1_score(
                y_true,
                y_pred,
                labels=observed_labels,
                average="micro",
                zero_division=0
            ),

            "cohen_kappa": kappa,

            "mcc": matthews_corrcoef(
                y_true, y_pred
            ),

            "correct_predictions": int(
                (y_true == y_pred).sum()
            ),

            "incorrect_predictions": int(
                (y_true != y_pred).sum()
            )
        }

    # Additional binary classification metrics
    if labels == [0, 1]:

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        metrics.update({

            "precision_positive": precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "recall_positive": recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "f1_positive": f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "specificity": (
                tn / (tn + fp)
                if (tn + fp) > 0 else np.nan
            ),

            "jaccard_positive": jaccard_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0
            ),

            "true_positive": tp,
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,

            "positive_support": int(
                (y_true == 1).sum()
            )
        })

    return metrics


# ============================================================
# CREATE IDENTICAL EVALUATION SAMPLES
# ============================================================

def prepare_evaluation(
    df,
    truth_column,
    prediction_columns,
    labels
):

    """Return or process prepare evaluation."""
    valid_truth = df[truth_column].isin(labels)

    valid_predictions = pd.DataFrame({
        model: df[column].isin(labels)
        for model, column in prediction_columns.items()
    })

    if USE_MATCHED_SAMPLE:

        common_mask = (
            valid_truth
            & valid_predictions.all(axis=1)
        )

        return {
            model: df.loc[common_mask].copy()
            for model in prediction_columns
        }

    return {
        model: df.loc[
            valid_truth & valid_predictions[model]
        ].copy()
        for model in prediction_columns
    }


print("Evaluation functions loaded.")

In [ ]:
# ============================================================
# VALIDATION ENGINE
# ============================================================

summary_results = []
class_results = []
confusion_matrices = {}
coverage_results = []

MODELS = list(model_datasets.keys())


def evaluate_task(
    df,
    task_name,
    truth_column,
    prediction_columns,
    labels,
    feature="ALL"
):

    """Return or process evaluate task."""
    evaluation_samples = prepare_evaluation(
        df,
        truth_column,
        prediction_columns,
        labels
    )

    for model, prediction_column in prediction_columns.items():

        sample = evaluation_samples[model]

        coverage_results.append({
            "task": task_name,
            "feature": feature,
            "model": model,
            "available_rows": len(df),
            "valid_human_labels": int(
                df[truth_column].isin(labels).sum()
            ),
            "valid_model_predictions": int(
                df[prediction_column].isin(labels).sum()
            ),
            "evaluated_rows": len(sample)
        })

        if sample.empty:
            continue

        y_true = sample[truth_column].tolist()
        y_pred = sample[prediction_column].tolist()

        if labels == BINARY_LABELS:

            y_true = [int(x) for x in y_true]
            y_pred = [int(x) for x in y_pred]

        metrics = calculate_metrics(
            y_true,
            y_pred,
            labels
        )

        summary_results.append({
            "task": task_name,
            "feature": feature,
            "model": model,
            **metrics
        })

        # Class-specific metrics
        report = classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=[str(x) for x in labels],
            output_dict=True,
            zero_division=0
        )

        for label in labels:

            values = report[str(label)]

            class_results.append({
                "task": task_name,
                "feature": feature,
                "model": model,
                "class": str(label),
                "precision": values["precision"],
                "recall": values["recall"],
                "f1_score": values["f1-score"],
                "support": values["support"]
            })

        confusion_matrices[
            (task_name, feature, model)
        ] = confusion_matrix(
            y_true,
            y_pred,
            labels=labels
        )


overall_columns = {
    model: f"{model_slugs[model]}_overall"
    for model in MODELS
}

binary_columns = {
    model: f"{model_slugs[model]}_binary"
    for model in MODELS
}

sentiment_columns = {
    model: f"{model_slugs[model]}_sentiment"
    for model in MODELS
}

# ============================================================
# TASK 1 — OVERALL SENTIMENT
# ============================================================

evaluate_task(
    df_validation_overall,
    "Overall sentiment",
    "manual_overall_sentiment",
    overall_columns,
    OVERALL_LABELS
)

# ============================================================
# TASK 2 — ASPECT IDENTIFICATION
# ============================================================

evaluate_task(
    df_validation_aspect,
    "Aspect identification",
    "manual_classification",
    binary_columns,
    BINARY_LABELS
)

# ============================================================
# TASK 3 — END-TO-END ABSA
# ============================================================

evaluate_task(
    df_validation_aspect,
    "End-to-end ABSA",
    "manual_sentiment",
    sentiment_columns,
    ASPECT_LABELS
)

# ============================================================
# TASK 4 — ABSA ON HUMAN-MENTIONED ASPECTS
# ============================================================

df_mentioned = df_validation_aspect.loc[
    df_validation_aspect["manual_classification"] == 1
].copy()

evaluate_task(
    df_mentioned,
    "ABSA on mentioned aspects",
    "manual_sentiment",
    sentiment_columns,
    ASPECT_LABELS
)

# ============================================================
# TASK 5 — CONDITIONAL SENTIMENT
# Both human and all evaluated models identified the aspect
# ============================================================

conditional_mask = (
    df_validation_aspect["manual_classification"] == 1
)

for column in binary_columns.values():

    conditional_mask &= (
        df_validation_aspect[column] == 1
    )

df_conditional = df_validation_aspect.loc[
    conditional_mask
].copy()

evaluate_task(
    df_conditional,
    "Conditional sentiment",
    "manual_sentiment",
    sentiment_columns,
    OVERALL_LABELS
)

# ============================================================
# TASK 6 — EXPERIENCE FEATURES ONLY
# Exclude the six journey-context descriptors
# ============================================================

df_experience = df_validation_aspect.loc[
    df_validation_aspect["feature"].isin(
        EXPERIENCE_FEATURES
    )
].copy()

evaluate_task(
    df_experience,
    "Experience aspect identification",
    "manual_classification",
    binary_columns,
    BINARY_LABELS
)

evaluate_task(
    df_experience,
    "Experience ABSA",
    "manual_sentiment",
    sentiment_columns,
    ASPECT_LABELS
)

# ============================================================
# TASK 7 — INDIVIDUAL FEATURE VALIDATION
# ============================================================

for feature in FEATURES:

    df_feature = df_validation_aspect.loc[
        df_validation_aspect["feature"] == feature
    ].copy()

    evaluate_task(
        df_feature,
        "Aspect identification",
        "manual_classification",
        binary_columns,
        BINARY_LABELS,
        feature=feature
    )

    evaluate_task(
        df_feature,
        "End-to-end ABSA",
        "manual_sentiment",
        sentiment_columns,
        ASPECT_LABELS,
        feature=feature
    )

    evaluate_task(
        df_feature.loc[
            df_feature["manual_classification"] == 1
        ],
        "ABSA on mentioned aspects",
        "manual_sentiment",
        sentiment_columns,
        ASPECT_LABELS,
        feature=feature
    )

# ============================================================
# CONSOLIDATE RESULTS
# ============================================================

df_metrics = pd.DataFrame(summary_results)

df_class_metrics = pd.DataFrame(class_results)

df_coverage = pd.DataFrame(coverage_results)

df_summary = df_metrics.loc[
    df_metrics["feature"] == "ALL"
].copy()

df_feature_metrics = df_metrics.loc[
    df_metrics["feature"] != "ALL"
].copy()

print("=" * 65)
print("VALIDATION COMPLETED")
print("=" * 65)

print("Metric records:", len(df_metrics))
print("Class-level records:", len(df_class_metrics))
print("Confusion matrices:", len(confusion_matrices))

display(
    df_summary[
        [
            "task",
            "model",
            "n",
            "accuracy",
            "balanced_accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "f1_weighted",
            "cohen_kappa",
            "mcc"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# MODEL COMPARISON TABLES
# ============================================================

KEY_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
    "cohen_kappa",
    "mcc"
]

for task in df_summary["task"].unique():

    print("\n" + "=" * 75)
    print(task.upper())
    print("=" * 75)

    table = (
        df_summary.loc[
            df_summary["task"] == task
        ]
        .set_index("model")[
            ["n"] + KEY_METRICS
        ]
        .sort_values(
            "f1_macro",
            ascending=False
        )
    )

    display(
        table.style
        .format({
            col: "{:.3f}"
            for col in KEY_METRICS
        })
        .highlight_max(
            subset=KEY_METRICS,
            color="#C6EFCE"
        )
    )

# ============================================================
# OVERALL MODEL RANKING PER TASK
# ============================================================

df_ranking = df_summary[
    ["task", "model", "n", "f1_macro"]
].copy()

df_ranking["rank"] = (
    df_ranking.groupby("task")["f1_macro"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

display(
    df_ranking.sort_values(
        ["task", "rank"]
    )
)

In [ ]:
# ============================================================
# MODEL PERFORMANCE CHARTS
# ============================================================

sns.set_theme(
    style="whitegrid",
    font_scale=1.05
)

chart_metrics = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro"
]

metric_names = {
    "accuracy": "Accuracy",
    "precision_macro": "Precision",
    "recall_macro": "Recall",
    "f1_macro": "Macro F1"
}

for task in df_summary["task"].unique():

    df_plot = df_summary.loc[
        df_summary["task"] == task,
        ["model"] + chart_metrics
    ].melt(
        id_vars="model",
        var_name="metric",
        value_name="score"
    )

    df_plot["metric"] = df_plot["metric"].map(
        metric_names
    )

    fig, ax = plt.subplots(figsize=(12, 6))

    sns.barplot(
        data=df_plot,
        x="metric",
        y="score",
        hue="model",
        palette=MODEL_COLORS,
        ax=ax
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt="%.3f",
            padding=3,
            fontsize=10
        )

    ax.set_ylim(0, 1.12)

    ax.set_title(
        f"Model Validation — {task}",
        fontsize=16,
        fontweight="bold",
        pad=15
    )

    ax.set_xlabel("")
    ax.set_ylabel("Score")

    ax.legend(
        title="Model",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=len(MODELS)
    )

    sns.despine()

    plt.tight_layout()

    filename = (
        task.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    fig.savefig(
        CHART_DIR / f"{filename}_comparison.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# CONFUSION MATRIX VISUALIZATIONS
# ============================================================

TASK_LABELS = {
    "Overall sentiment": OVERALL_LABELS,
    "Aspect identification": BINARY_LABELS,
    "End-to-end ABSA": ASPECT_LABELS,
    "ABSA on mentioned aspects": ASPECT_LABELS,
    "Conditional sentiment": OVERALL_LABELS,
    "Experience aspect identification": BINARY_LABELS,
    "Experience ABSA": ASPECT_LABELS
}

for task, labels in TASK_LABELS.items():

    available_models = [
        model for model in MODELS
        if (task, "ALL", model) in confusion_matrices
    ]

    if not available_models:
        continue

    fig, axes = plt.subplots(
        1,
        len(available_models),
        figsize=(6 * len(available_models), 5.5),
        squeeze=False
    )

    for ax, model in zip(
        axes.ravel(),
        available_models
    ):

        matrix = confusion_matrices[
            (task, "ALL", model)
        ]

        row_totals = matrix.sum(
            axis=1,
            keepdims=True
        )

        normalized = np.divide(
            matrix,
            row_totals,
            out=np.zeros_like(
                matrix,
                dtype=float
            ),
            where=row_totals != 0
        )

        annotations = np.empty(
            matrix.shape,
            dtype=object
        )

        for i in range(matrix.shape[0]):

            for j in range(matrix.shape[1]):

                annotations[i, j] = (
                    f"{matrix[i, j]}\n"
                    f"({normalized[i, j]:.1%})"
                )

        sns.heatmap(
            normalized,
            annot=annotations,
            fmt="",
            cmap="Blues",
            vmin=0,
            vmax=1,
            square=True,
            cbar=False,
            xticklabels=labels,
            yticklabels=labels,
            ax=ax
        )

        ax.set_title(
            model,
            fontweight="bold"
        )

        ax.set_xlabel("Predicted label")
        ax.set_ylabel("Human annotation")

        ax.tick_params(
            axis="x",
            rotation=45
        )

        ax.tick_params(
            axis="y",
            rotation=0
        )

    fig.suptitle(
        f"Confusion Matrices — {task}",
        fontsize=16,
        fontweight="bold"
    )

    plt.tight_layout()

    filename = (
        task.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    fig.savefig(
        CHART_DIR / f"{filename}_confusion_matrices.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# FEATURE-LEVEL PERFORMANCE HEATMAPS
# ============================================================

feature_tasks = {
    "Aspect identification": "f1_positive",
    "End-to-end ABSA": "f1_macro",
    "ABSA on mentioned aspects": "f1_macro"
}

for task, metric in feature_tasks.items():

    df_plot = df_feature_metrics.loc[
        df_feature_metrics["task"] == task
    ].copy()

    if df_plot.empty:
        continue

    # Do not interpret positive-class F1 for aspects
    # with no positive examples in the human sample.
    if task == "Aspect identification":

        df_plot.loc[
            df_plot["positive_support"] == 0,
            metric
        ] = np.nan

    table = df_plot.pivot(
        index="feature",
        columns="model",
        values=metric
    )

    table = table.reindex(
        index=FEATURES,
        columns=MODELS
    )

    table.index = (
        table.index
        .str.replace("_", " ")
        .str.title()
    )

    fig, ax = plt.subplots(
        figsize=(9, 13)
    )

    sns.heatmap(
        table,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        mask=table.isna(),
        cbar_kws={
            "label": metric.replace("_", " ").title()
        },
        ax=ax
    )

    ax.set_title(
        f"{task}\nFeature-level F1 scores",
        fontsize=16,
        fontweight="bold",
        pad=15
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.tick_params(
        axis="x",
        rotation=0
    )

    plt.tight_layout()

    filename = (
        task.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    fig.savefig(
        CHART_DIR / f"{filename}_feature_heatmap.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# REVIEW-LEVEL MULTILABEL VALIDATION
# ============================================================

multilabel_results = []

common_binary = prepare_evaluation(
    df_validation_aspect,
    "manual_classification",
    binary_columns,
    BINARY_LABELS
)

for model in MODELS:

    sample = common_binary[model]

    prediction_col = binary_columns[model]

    y_true = sample.pivot(
        index="record_key",
        columns="feature",
        values="manual_classification"
    ).reindex(columns=FEATURES)

    y_pred = sample.pivot(
        index="record_key",
        columns="feature",
        values=prediction_col
    ).reindex(columns=FEATURES)

    # Require all 25 classifications for each review
    complete = (
        y_true.notna().all(axis=1)
        & y_pred.notna().all(axis=1)
    )

    y_true = y_true.loc[complete].astype(int)
    y_pred = y_pred.loc[complete].astype(int)

    if y_true.empty:
        continue

    multilabel_results.append({

        "model": model,

        "n_reviews": len(y_true),

        # All 25 features must be correct
        "exact_match_accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "hamming_accuracy": float(
            (
                y_true.to_numpy()
                == y_pred.to_numpy()
            ).mean()
        ),

        "precision_micro": precision_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),

        "recall_micro": recall_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),

        "f1_micro": f1_score(
            y_true,
            y_pred,
            average="micro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "f1_samples": f1_score(
            y_true,
            y_pred,
            average="samples",
            zero_division=0
        ),

        "jaccard_samples": jaccard_score(
            y_true,
            y_pred,
            average="samples",
            zero_division=0
        ),

        "avg_human_aspects": (
            y_true.sum(axis=1).mean()
        ),

        "avg_predicted_aspects": (
            y_pred.sum(axis=1).mean()
        )
    })

df_multilabel = pd.DataFrame(
    multilabel_results
)

display(df_multilabel.round(4))

In [ ]:
# ============================================================
# REVIEW-CLUSTERED BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

N_BOOTSTRAPS = 2000
RANDOM_SEED = 42

bootstrap_results = []
bootstrap_distributions = {}

bootstrap_tasks = [
    (
        "Overall sentiment",
        df_validation_overall,
        "manual_overall_sentiment",
        overall_columns,
        OVERALL_LABELS
    ),
    (
        "Aspect identification",
        df_validation_aspect,
        "manual_classification",
        binary_columns,
        BINARY_LABELS
    ),
    (
        "End-to-end ABSA",
        df_validation_aspect,
        "manual_sentiment",
        sentiment_columns,
        ASPECT_LABELS
    ),
    (
        "ABSA on mentioned aspects",
        df_mentioned,
        "manual_sentiment",
        sentiment_columns,
        ASPECT_LABELS
    )
]

for task, df, truth_col, pred_cols, labels in bootstrap_tasks:

    samples = prepare_evaluation(
        df,
        truth_col,
        pred_cols,
        labels
    )

    print(f"Bootstrapping: {task}")

    for model, sample in samples.items():

        if sample.empty:
            continue

        sample = sample.reset_index(drop=True)

        # Reset seed so matched models use identical resamples
        rng = np.random.default_rng(RANDOM_SEED)

        groups = list(
            sample.groupby(
                "record_key",
                sort=True
            ).indices.values()
        )

        y_true = sample[truth_col].to_numpy()
        y_pred = sample[pred_cols[model]].to_numpy()

        if labels == BINARY_LABELS:

            y_true = y_true.astype(int)
            y_pred = y_pred.astype(int)

        else:

            y_true = y_true.astype(str)
            y_pred = y_pred.astype(str)

        fixed_labels = [
            label for label in labels
            if label in y_true or label in y_pred
        ]

        accuracy_values = []
        f1_values = []

        for _ in range(N_BOOTSTRAPS):

            selected_groups = rng.integers(
                0,
                len(groups),
                size=len(groups)
            )

            selected_indices = np.concatenate([
                groups[i]
                for i in selected_groups
            ])

            yt = y_true[selected_indices]
            yp = y_pred[selected_indices]

            accuracy_values.append(
                accuracy_score(yt, yp)
            )

            f1_values.append(
                f1_score(
                    yt,
                    yp,
                    labels=fixed_labels,
                    average="macro",
                    zero_division=0
                )
            )

        for metric, values, point_estimate in [

            (
                "accuracy",
                accuracy_values,
                accuracy_score(y_true, y_pred)
            ),

            (
                "f1_macro",
                f1_values,
                f1_score(
                    y_true,
                    y_pred,
                    labels=fixed_labels,
                    average="macro",
                    zero_division=0
                )
            )
        ]:

            bootstrap_distributions[
                (task, model, metric)
            ] = np.asarray(values)

            bootstrap_results.append({
                "task": task,
                "model": model,
                "metric": metric,
                "n_reviews": len(groups),
                "n_observations": len(sample),
                "score": point_estimate,
                "ci_lower_95": np.percentile(
                    values, 2.5
                ),
                "ci_upper_95": np.percentile(
                    values, 97.5
                )
            })

df_bootstrap = pd.DataFrame(
    bootstrap_results
)

display(df_bootstrap.round(4))

In [ ]:
# ============================================================
# MODEL-TO-MODEL AGREEMENT
# ============================================================

from itertools import combinations

agreement_results = []

for task, df, truth_col, pred_cols, labels in bootstrap_tasks:

    for model_a, model_b in combinations(MODELS, 2):

        col_a = pred_cols[model_a]
        col_b = pred_cols[model_b]

        valid = (
            df[truth_col].isin(labels)
            & df[col_a].isin(labels)
            & df[col_b].isin(labels)
        )

        sample = df.loc[valid].copy()

        if sample.empty:
            continue

        a = sample[col_a].tolist()
        b = sample[col_b].tolist()

        if labels == BINARY_LABELS:

            a = [int(x) for x in a]
            b = [int(x) for x in b]

        agreement_results.append({
            "task": task,
            "model_a": model_a,
            "model_b": model_b,
            "n": len(sample),
            "agreement": accuracy_score(a, b),
            "cohen_kappa": cohen_kappa_score(
                a, b, labels=labels
            )
        })

df_agreement = pd.DataFrame(
    agreement_results
)

display(df_agreement.round(4))


# ============================================================
# PAIRED BOOTSTRAP DIFFERENCES
# ============================================================

difference_results = []

if USE_MATCHED_SAMPLE:

    for task, *_ in bootstrap_tasks:

        for model_a, model_b in combinations(MODELS, 2):

            for metric in ["accuracy", "f1_macro"]:

                key_a = (task, model_a, metric)
                key_b = (task, model_b, metric)

                if (
                    key_a not in bootstrap_distributions
                    or key_b not in bootstrap_distributions
                ):
                    continue

                differences = (
                    bootstrap_distributions[key_a]
                    - bootstrap_distributions[key_b]
                )

                lower, upper = np.percentile(
                    differences,
                    [2.5, 97.5]
                )

                difference_results.append({

                    "task": task,
                    "metric": metric,
                    "model_a": model_a,
                    "model_b": model_b,

                    "mean_difference_pp": (
                        differences.mean() * 100
                    ),

                    "ci_lower_pp": lower * 100,

                    "ci_upper_pp": upper * 100,

                    "ci_excludes_zero": (
                        lower > 0 or upper < 0
                    )
                })

df_model_differences = pd.DataFrame(
    difference_results
)

display(df_model_differences.round(3))

In [ ]:
# ============================================================
# OVERALL SENTIMENT ERRORS
# ============================================================

overall_errors = []
aspect_errors = []

for model in MODELS:

    slug = model_slugs[model]

    pred_col = f"{slug}_overall"

    valid = (
        df_validation_overall[
            "manual_overall_sentiment"
        ].isin(OVERALL_LABELS)
        &
        df_validation_overall[
            pred_col
        ].isin(OVERALL_LABELS)
    )

    errors = df_validation_overall.loc[
        valid
        &
        (
            df_validation_overall[
                "manual_overall_sentiment"
            ]
            != df_validation_overall[pred_col]
        )
    ].copy()

    errors["model"] = model

    errors["predicted_sentiment"] = errors[pred_col]

    overall_errors.append(
        errors[
            [
                "record_key",
                "airline",
                "source",
                "review_date",
                "model",
                "manual_overall_sentiment",
                "predicted_sentiment"
            ]
        ]
    )

    # ========================================================
    # ASPECT CLASSIFICATION ERRORS
    # ========================================================

    binary_col = f"{slug}_binary"
    sentiment_col = f"{slug}_sentiment"

    valid_binary = (
        df_validation_aspect[
            "manual_classification"
        ].isin(BINARY_LABELS)
        &
        df_validation_aspect[
            binary_col
        ].isin(BINARY_LABELS)
    )

    valid_sentiment = (
        df_validation_aspect[
            "manual_sentiment"
        ].isin(ASPECT_LABELS)
        &
        df_validation_aspect[
            sentiment_col
        ].isin(ASPECT_LABELS)
    )

    binary_error = (
        valid_binary
        &
        (
            df_validation_aspect[
                "manual_classification"
            ]
            != df_validation_aspect[binary_col]
        )
    )

    sentiment_error = (
        valid_sentiment
        &
        (
            df_validation_aspect[
                "manual_sentiment"
            ]
            != df_validation_aspect[sentiment_col]
        )
    )

    errors = df_validation_aspect.loc[
        binary_error | sentiment_error
    ].copy()

    errors["model"] = model

    errors["predicted_classification"] = (
        errors[binary_col]
    )

    errors["predicted_sentiment"] = (
        errors[sentiment_col]
    )

    errors["binary_error"] = binary_error.loc[
        errors.index
    ]

    errors["sentiment_error"] = sentiment_error.loc[
        errors.index
    ]

    errors["error_type"] = np.select(
        [
            errors["binary_error"]
            & (
                errors["manual_classification"] == 1
            ),

            errors["binary_error"]
            & (
                errors["manual_classification"] == 0
            ),

            errors["sentiment_error"]
        ],
        [
            "Missed aspect",
            "False aspect detection",
            "Incorrect sentiment"
        ],
        default="Other"
    )

    aspect_errors.append(
        errors[
            [
                "record_key",
                "airline",
                "source",
                "feature",
                "model",
                "manual_classification",
                "predicted_classification",
                "manual_sentiment",
                "predicted_sentiment",
                "binary_error",
                "sentiment_error",
                "error_type"
            ]
        ]
    )

df_overall_errors = pd.concat(
    overall_errors,
    ignore_index=True
)

df_aspect_errors = pd.concat(
    aspect_errors,
    ignore_index=True
)

# Add original review text
if "review_en" in df_hf.columns:

    review_text = df_hf[
        ["record_key", "review_en"]
    ]

    df_overall_errors = df_overall_errors.merge(
        review_text,
        on="record_key",
        how="left"
    )

    df_aspect_errors = df_aspect_errors.merge(
        review_text,
        on="record_key",
        how="left"
    )

print("\nOverall sentiment errors:")

display(
    df_overall_errors.groupby("model")
    .size()
    .to_frame("errors")
)

print("\nAspect errors by model and type:")

display(
    pd.crosstab(
        df_aspect_errors["model"],
        df_aspect_errors["error_type"]
    )
)

print("\nFeatures with most classification errors:")

display(
    pd.crosstab(
        df_aspect_errors["feature"],
        df_aspect_errors["model"]
    )
)

In [ ]:
# ============================================================
# EXPORT CONFUSION MATRICES IN TABULAR FORMAT
# ============================================================

cm_records = []

for (
    task,
    feature,
    model
), matrix in confusion_matrices.items():

    labels = TASK_LABELS[task]

    for i, actual in enumerate(labels):

        for j, predicted in enumerate(labels):

            row_total = matrix[i].sum()

            cm_records.append({

                "task": task,
                "feature": feature,
                "model": model,

                "actual": str(actual),
                "predicted": str(predicted),

                "count": int(matrix[i, j]),

                "row_percentage": (
                    matrix[i, j] / row_total
                    if row_total > 0
                    else np.nan
                )
            })

df_confusion_export = pd.DataFrame(
    cm_records
)

# ============================================================
# EXPORT ALL RESULTS
# ============================================================

RESULTS_FILE = (
    OUTPUT_DIR / "model_validation_results.xlsx"
)

export_tables = {
    "Summary": df_summary,
    "Model Ranking": df_ranking,
    "Feature Metrics": df_feature_metrics,
    "Class Metrics": df_class_metrics,
    "Multilabel Metrics": df_multilabel,
    "Confidence Intervals": df_bootstrap,
    "Model Differences": df_model_differences,
    "Model Agreement": df_agreement,
    "Confusion Matrices": df_confusion_export,
    "Overall Errors": df_overall_errors,
    "Aspect Errors": df_aspect_errors,
    "Evaluation Coverage": df_coverage,
    "Filter Audit": df_filter_audit,
    "Overall Predictions": df_validation_overall,
    "Aspect Predictions": df_validation_aspect
}

with pd.ExcelWriter(
    RESULTS_FILE,
    engine="openpyxl"
) as writer:

    for sheet_name, table in export_tables.items():

        table.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes = "A2"

        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        # Adjust column widths
        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0].column_letter
            )

            max_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                max(max_length + 2, 12),
                65
            )

# ============================================================
# EXPORT CSV FILES
# ============================================================

for sheet_name, table in export_tables.items():

    filename = (
        sheet_name.lower()
        .replace(" ", "_")
    )

    table.to_csv(
        OUTPUT_DIR / f"{filename}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("=" * 70)
print("VALIDATION REPORT COMPLETED")
print("=" * 70)

print("Models evaluated:", ", ".join(MODELS))

print(
    "Eligible validation reviews:",
    len(df_validation_overall)
)

print(
    "Eligible review-feature pairs:",
    len(df_validation_aspect)
)

print("\nExcel report:")
print(RESULTS_FILE)

print("\nCharts:")
print(CHART_DIR)

print("\nCSV reports:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# PRESENTATION-READY VALIDATION SUMMARY
# ============================================================

presentation_metrics = [

    (
        "Overall Accuracy",
        "Overall sentiment",
        "accuracy"
    ),

    (
        "Overall Macro F1",
        "Overall sentiment",
        "f1_macro"
    ),

    (
        "Aspect Detection F1",
        "Aspect identification",
        "f1_positive"
    ),

    (
        "End-to-End ABSA Macro F1",
        "End-to-end ABSA",
        "f1_macro"
    ),

    (
        "Mentioned-Aspect Macro F1",
        "ABSA on mentioned aspects",
        "f1_macro"
    ),

    (
        "Overall Cohen's Kappa",
        "Overall sentiment",
        "cohen_kappa"
    )
]

presentation_records = []

for label, task, metric in presentation_metrics:

    for model in MODELS:

        result = df_summary.loc[
            (df_summary["task"] == task)
            & (df_summary["model"] == model)
        ]

        if result.empty:
            continue

        presentation_records.append({
            "metric": label,
            "model": model,
            "score": result.iloc[0][metric]
        })

df_presentation = (
    pd.DataFrame(presentation_records)
    .pivot(
        index="metric",
        columns="model",
        values="score"
    )
    .reindex(
        index=[
            x[0] for x in presentation_metrics
        ],
        columns=MODELS
    )
)

display(
    df_presentation.style
    .format("{:.3f}")
    .highlight_max(
        axis=1,
        color="#C6EFCE"
    )
)

df_presentation.to_excel(
    OUTPUT_DIR / "presentation_validation_summary.xlsx"
)